In [4]:
%run ../how_do_we_assemble_genomes_functions.ipynb

**String Composition Problem: Generate the k-mer composition of a string.**

In [17]:
with open("string_composition_problem.txt", "r") as f:
    lines = f.readlines()
    k = int(lines[0].strip())
    Text = lines[1].strip()

kmers = string_composition(Text, k)

with open("string_composition_problem_output.txt", "w") as g:
    g.write(" ".join(kmers))

**Reconstruct a string from its genome path**

In [4]:
path = open("genome_path_problem.txt").read()
genomefrompath = open("genome_path_problem_output.txt", "w")
genomefrompath.write(PathtoGenome(path))
genomefrompath.close()

**Overlap Graph Problem**

In [7]:
nodes = open("overlap_graph_problem.txt", "r").read()
with open("overlap_graph_problem_output.txt", 'w') as file:
    result_dict = OverlapGraph(nodes)
    for key, value in result_dict.items():
        formatted_value = ' '.join(value)
        file.write(f"{key}: {formatted_value}\n")

**Generating a k-universal binary string**

In [2]:
print(k_universal_binary_string(4))

0000100110101111000


**De Bruijn Graph from a String Problem**

In [ ]:
with open("de_bruijn_graph_from_a_string_problem.txt", "r") as f:
    data = f.readlines()
    k = int(data[0].strip())
    Text = data[1].strip()
    with open("de_bruijn_graph_from_a_string_problem_output.txt", 'w') as file:
        result_dict = DeBruijnGraph_from_string(Text, k)
        for key, value in result_dict.items():
            formatted_value = ' '.join(value)
            file.write(f"{key}: {formatted_value}\n")

**DeBruijn Graph from k-mers Problem**

In [2]:
kmers = open("de_bruijn_graph_from_kmers_problem.txt", "r").read().strip()
with open("de_bruijn_graph_from_kmers_problem_output.txt", 'w') as file:
    result_dict = DeBruijnGraph_from_kmers(kmers)
    for key, value in result_dict.items():
        formatted_value = ' '.join(value)
        file.write(f"{key}: {formatted_value}\n")

**Eulerian Cycle Problem:**

In [3]:
filename = "eulerian_cycle_problem.txt"
graph = read_adjacency_list(filename)

eulerian_cycle_result = eulerian_cycle(graph)
with open('eulerian_cycle_problem_output.txt', 'w') as output_data:
    output_data.write(' '.join(map(str, eulerian_cycle_result)))

**Eulerian Path Problem**

In [6]:
filename = "eulerian_path_problem.txt"
graph = read_adjacency_list(filename)

eulerian_path_result = eulerian_path(graph)
with open('eulerian_path_problem_output.txt', 'w') as output_data:
    output_data.write(' '.join(map(str, eulerian_path_result)))

The graph is unbalanced and will form an Eulerian Path, not an Eulerian Cycle


**String Reconstruction Problem: Combining all of the above**

In [108]:
from collections import defaultdict, OrderedDict
def DeBruijnGraph_from_kmers(kmers):
  de_bruijn_graph = defaultdict(list)
  for i in kmers.split():
    prefix = i[:len(i)-1]
    suffix = i[1:]
    de_bruijn_graph[prefix].append(suffix)
    
  return OrderedDict(de_bruijn_graph.items())

In [109]:
kmers = open("reads_string_reconstruction.txt", "r").readlines()[1]
with open("de_bruijn_graph_from_reads.txt", 'w') as file:
    result_dict = DeBruijnGraph_from_kmers(kmers)
    for key, value in result_dict.items():
        formatted_value = ' '.join(value)
        file.write(f"{key}: {formatted_value}\n")

file.close()

In [110]:
def read_adjacency_list(filename):
    graph = {}
    with open(filename, 'r') as file:
        for line in file:
            parts = line.strip().split(':')
            node = str(parts[0]) # str instead of int
            neighbors = list(map(str, parts[1].split()))
            graph[node] = neighbors
    return graph

filename = "de_bruijn_graph_from_reads.txt"
graph = read_adjacency_list(filename)

In [111]:
def check_balance(graph):
    incoming_edges = 0
    outgoing_edges = 0
    start_nodes = []
    for i, j in graph.items():
        outgoing_edges = len(j)
        for k in graph.values():
            if i in k:
                incoming_edges += 1
        if outgoing_edges > incoming_edges:
            start_nodes.append(i)
        incoming_edges = 0

    return start_nodes

In [112]:
import random

def eulerian_path(graph):
    if not graph:
        raise ValueError("The graph is empty.")

    start_nodes  = check_balance(graph) # type: ignore
    if len(start_nodes) > 0:
        print("The graph is unbalanced and will form an Eulerian Path, not an Eulerian Cycle")
        current_node = random.choice(start_nodes)
        path = [current_node]
        
        while True:
            if current_node not in graph or not graph[current_node]:
                break
            next_node = graph[current_node][0]
            path.append(next_node)
            
            if len(graph[current_node]) == 1:
                del graph[current_node]
            else:
                graph[current_node] = graph[current_node][1:]
            
            current_node = next_node

        while len(graph) > 0:
            for i in range(len(path)):
                if path[i] in graph:
                    current_node = path[i]
                    cycle = [current_node]
                    while True:
                        if current_node not in graph or not graph[current_node]:
                            break
                        next_node = graph[current_node][0]
                        cycle.append(next_node)

                        if len(graph[current_node]) == 1:
                            del graph[current_node]
                        else:
                            graph[current_node] = graph[current_node][1:]
                        
                        current_node = next_node

                    path = path[:i] + cycle + path[i+1:]
                    break

        return path
    else:
        print("The graph is balanced and will form an Eulerian Cycle, not an Eulerian Path")
        return eulerian_cycle(graph)

In [113]:
eulerian_path_result = eulerian_path(graph)
with open('eulerian_path_from_debruijn_graph.txt', 'w') as output_data:
    output_data.write(' '.join(map(str, eulerian_path_result)))

output_data.close()

The graph is unbalanced and will form an Eulerian Path, not an Eulerian Cycle


In [117]:
directed_reads = open('eulerian_path_from_debruijn_graph.txt', 'r').read().split()
string_reconstruction = directed_reads[0]
for i in directed_reads[1:]:
    string_reconstruction += i[-1]

with open('genome_reconstructed.txt', 'w') as output_data:
    output_data.write(''.join(map(str, string_reconstruction)))

output_data.close()

**k-Universal Circular String Problem**

In [34]:
def generate_binary_kmers(k):
    if k == 0:
        return ['']
    else:
        kmers_with_zero = [kmer + '0' for kmer in generate_binary_kmers(k - 1)]
        kmers_with_one = [kmer + '1' for kmer in generate_binary_kmers(k - 1)]
        return kmers_with_zero + kmers_with_one

k = 9
binary_kmers = generate_binary_kmers(k)

In [35]:
from collections import defaultdict, OrderedDict
def DeBruijnGraph_from_kmers(kmers):
  de_bruijn_graph = defaultdict(list)
  for i in kmers:
    prefix = i[:len(i)-1]
    suffix = i[1:]
    de_bruijn_graph[prefix].append(suffix)
    
  return OrderedDict(de_bruijn_graph.items())

In [36]:
import random

def eulerian_cycle(graph):
    if not graph:
        raise ValueError("The graph is empty.")

    current_node = random.choice(list(graph.keys()))
    path = [current_node]
    
    while True:
        if current_node not in graph or not graph[current_node]:
            break
        next_node = graph[current_node][0]
        path.append(next_node)
        
        if len(graph[current_node]) == 1:
            del graph[current_node]
        else:
            graph[current_node] = graph[current_node][1:]
        
        current_node = next_node

    while len(graph) > 0:
        for i in range(len(path)):
            if path[i] in graph:
                current_node = path[i]
                cycle = [current_node]
                while True:
                    if current_node not in graph or not graph[current_node]:
                        break
                    next_node = graph[current_node][0]
                    cycle.append(next_node)

                    if len(graph[current_node]) == 1:
                        del graph[current_node]
                    else:
                        graph[current_node] = graph[current_node][1:]
                    
                    current_node = next_node

                path = path[:i] + cycle + path[i+1:]
                break

    return path

In [37]:
debruijn_graph = DeBruijnGraph_from_kmers(binary_kmers)
cycle = eulerian_cycle(debruijn_graph)
circular_string = cycle[0]
for i in cycle[1:-k + 1]: # As it is a circular string, we need not require the nodes of the entire cycle
    circular_string += i[-1]

with open('circular_string.txt', 'w') as output_file:
    output_file.write(circular_string)

output_file.close()

**Generate (k,d)-mer composition of a string**

In [10]:
def paired_composition(Text, k, d):
    read_pairs = []
    for i in range(len(Text) - 2 * k):
        read_pairs.append("(" + Text[i:i+k] + "|" + Text[i+k+d:i+2*k+d] + ")")

    return ''.join(sorted(read_pairs))

Text = "TAATGCCATGGGATGTT"
k = 3
d = 2
paired_composition(Text, k, d)

'(AAT|CAT)(ATG|ATG)(ATG|ATG)(CAT|GAT)(CCA|GGA)(GCC|GGG)(GGA|TT)(GGG|GTT)(TAA|CCA)(TGC|TGG)(TGG|TGT)'